# Paper Figure Reproduction (DDI-Classifier)

This notebook reproduces the **evaluation / analysis figures** of the paper
from the **final datasets only** (no intermediate pipeline data needed).

| Cell | Figure | Content |
|------|--------|---------|
| Setup | - | paths, imports, plotting style, data loading |
| Fig 1D | **Figure 1D** | Crosslink retention after domain splitting |
| Fig 3A | **Figure 3A** | ROC of key structural scores + RF models (test set) |
| Fig 3B | **Figure 3B** | Recall at FDR=5% (1:1 sampling), all feature scores |
| Fig 3C | **Figure 3C** | Recall at FDR=5% across ratios 1:1-1:128 (core 4 scores) |
| Fig 3D | **Figure 3D** | Fixed-threshold recall scatter at 1:128 subset |
| Fig 4A/4B | **Figure 4A/4B** | Gini importance (SPOC ESMFOLD / Structural classifier) |
| Supplementary | - | AUPR (Precision-Recall) of the 3 models |
| Supplementary | - | RF hyperparameter sweep (3-fold CV AUPR heatmaps) |

**How to run**
1. `pip install -r requirements.txt` at the repository root.
2. Start jupyter from the `notebooks/` directory (or open this file in VS Code).
3. Run all cells top to bottom. The Setup cell auto-points to `data/analysis/`.

> To override the repo root, set `CLASSIFIER_PACKAGE=/path/to/classifier_package`.
> Figures are written to `figures/` at the repo root (created automatically).
> Figure 3D reuses the 1:128 sampled subsets built by the Figure 3C cell -
> run cells in order.

In [ ]:
# ============================================================
# Setup: paths, imports, plotting style, data loading
# ------------------------------------------------------------
# All data used by this notebook lives under data/analysis/
# (shipped with the repository).
# ============================================================
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, MaxNLocator
from sklearn.metrics import (roc_curve, auc,
                             precision_recall_curve, average_precision_score)

# ---- paths ------------------------------------------------------
# REPO: repository root. Use env CLASSIFIER_PACKAGE if set, else the
#       parent of the current working directory (i.e. run jupyter from notebooks/).
REPO = os.environ.get('CLASSIFIER_PACKAGE',
                      os.path.abspath(os.path.join(os.getcwd(), '..')))
DATA    = os.path.join(REPO, 'data', 'analysis')   # plotting data shipped with the repo
FIG_DIR = os.path.join(REPO, 'figures')             # figure output dir (repo root)
os.makedirs(FIG_DIR, exist_ok=True)

# ---- plotting style (paper: white background, black axes) -------
plt.rcParams.update({
    'font.size': 12,
    'axes.grid': False,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'pdf.fonttype': 42,          # TrueType embedding -> editable text in Adobe
    'ps.fonttype': 42,
    'text.color': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'axes.edgecolor': 'black',
    'xtick.bottom': True,
    'ytick.left': True,
    'xtick.top': False,
    'ytick.right': False,
    'xtick.major.size': 3.5,
    'ytick.major.size': 3.5,
})

# ---- load test / train ------------------------------------------
df_test  = pd.read_csv(f'{DATA}/test.tsv', sep='\t')
df_train = pd.read_csv(f'{DATA}/train.tsv', sep='\t')
y_test = df_test['label'].values

# ---- RF model predictions on test set ----------------------------
def load_rf_prediction(path, df):
    """Load a pickled RF (dict with 'model'/'imputer'/'features') and predict."""
    with open(path, 'rb') as f:
        m = pickle.load(f)
    X = m['imputer'].transform(df[m['features']].values)
    return m['model'].predict_proba(X)[:, 1]

y_prob_all    = load_rf_prediction(f'{DATA}/rf_all_feat_model.pkl', df_test)
y_prob_struct = load_rf_prediction(f'{DATA}/rf_struct_feat_model.pkl', df_test)

# ---- degree-match baseline (same feature def as model_baseline.py) --
def build_degree_maps(df_tr):
    """Degree per uniprot from train only (no test labels, avoids circularity)."""
    pos_deg, all_deg = {}, {}
    for _, r in df_tr[df_tr['label'] == 1].iterrows():
        pos_deg.setdefault(r['uniprot_A'], set()).add(r['uniprot_B'])
        pos_deg.setdefault(r['uniprot_B'], set()).add(r['uniprot_A'])
    for _, r in df_tr.iterrows():
        all_deg.setdefault(r['uniprot_A'], set()).add(r['uniprot_B'])
        all_deg.setdefault(r['uniprot_B'], set()).add(r['uniprot_A'])
    return ({k: len(v) for k, v in pos_deg.items()},
            {k: len(v) for k, v in all_deg.items()})

def build_degree_features(df, pos_deg, all_deg):
    """14 pure-degree features (column order matches model_baseline.py)."""
    aP = df['uniprot_A'].map(pos_deg).fillna(0).values.astype(float)
    bP = df['uniprot_B'].map(pos_deg).fillna(0).values.astype(float)
    aA = df['uniprot_A'].map(all_deg).fillna(0).values.astype(float)
    bA = df['uniprot_B'].map(all_deg).fillna(0).values.astype(float)
    return np.column_stack([aP, bP, aA, bA, aP + bP, np.abs(aP - bP),
                            np.minimum(aP, bP), np.maximum(aP, bP), aP * bP,
                            aA + bA, np.abs(aA - bA), np.minimum(aA, bA),
                            np.maximum(aA, bA), aA * bA])

def load_degree_prediction(df):
    """Degree-match RF probabilities on df."""
    with open(f'{DATA}/rf_degree_match_model.pkl', 'rb') as f:
        dm = pickle.load(f)
    pos_deg, all_deg = build_degree_maps(df_train)
    X = build_degree_features(df, pos_deg, all_deg)
    return dm['model'].predict_proba(X)[:, 1]

y_prob_degree = load_degree_prediction(df_test)

# ---- shared score dictionaries (used by Fig3A-3D) ----------------
SCORES = {
    'all_feature_RF': y_prob_all,                        # RF all-feature
    'structure_RF':   y_prob_struct,                     # RF structure-only
    'pdockq':         df_test['pdockq_e'].values,        # pDockQ
    'pdockq2':        df_test['pdockq_e_v2'].values,     # pDockQ v2
    'iptm':           df_test['iptm'].values,            # ipTM
    'ipsae':          df_test['ipsae_d0res_asym'].values,# ipSAE (per-residue d0)
}
LABEL_NAME = {'all_feature_RF': 'SPOC ESMFOLD', 'structure_RF': 'Structural classifier',
              'pdockq': 'pDockQ', 'pdockq2': 'pDockQ v2',
              'iptm': 'ipTM', 'ipsae': 'ipSAE'}
COLORS = {'all_feature_RF': '#1B5E20', 'structure_RF': '#0D47A1',
          'pdockq': '#00695C', 'pdockq2': '#E65100',
          'iptm': '#6A1B9A', 'ipsae': '#F57F17'}
CORE_SCORE_KEYS = ['all_feature_RF', 'structure_RF', 'iptm', 'ipsae']  # Fig3C/3D

print(f'REPO = {REPO}')
print(f'DATA = {DATA}')
print(f'test : {len(df_test)} pairs (pos={int(y_test.sum())})')
print(f'train: {len(df_train)} pairs')


## Figure 1D — Crosslink retention after domain splitting

Histogram of the fraction of crosslinks retained inside split domains
($n_{dom}/n_{total}$) for positive (interacting) domain pairs. Counts are shown
on a manual $\log_{10}$ scale (linear axis with $10^k$ tick labels) so that the
zero-count bars render correctly in vector PDFs.

In [ ]:
# ============================================================
# Figure 1D — Crosslink retention after domain splitting
# ------------------------------------------------------------
# Data: data/analysis/domain_pairs_stats.csv
# (pre-extracted from the pipeline's domain_pairs.pkl, positive
#  pairs only, to avoid shipping the large intermediate file).
# ============================================================
df_dp = pd.read_csv(f'{DATA}/domain_pairs_stats.csv')

cl_domain = df_dp['n_crosslinks_domain'].to_numpy(float)
cl_total = df_dp['n_crosslinks_total'].to_numpy(float)
ratio = np.divide(cl_domain, cl_total,
                  out=np.zeros_like(cl_domain, dtype=float), where=cl_total > 0)

n_domain_pairs = len(df_dp)          # positive domain pairs
n_crosslinks = int(cl_total.sum())   # crosslinks/interactions involved
frac_full = np.mean(np.isclose(ratio, 1.0))

# manual log10 of counts (avoid ax.set_yscale('log') which can corrupt
# zero-count bars in vector PDFs); y-axis stays linear with 10^k labels.
counts, edges = np.histogram(ratio, bins=50)
centers = 0.5 * (edges[:-1] + edges[1:])
with np.errstate(divide='ignore'):
    log_counts = np.log10(counts.astype(float))
log_counts[counts == 0] = np.nan

fig, ax = plt.subplots(figsize=(8, 4), facecolor='white')
ax.bar(centers, log_counts, width=np.diff(edges),
       color='skyblue', edgecolor='white', linewidth=1.2, alpha=0.9)
ax.axvline(np.mean(ratio), color='black', ls='--', lw=1.5,
           label=f'Mean = {np.mean(ratio):.1%}')

ax.set_xlabel('Retained crosslinks ratio  ($n_{dom}/n_{total}$)')
ax.set_ylabel(r'$\log_{10}$(Count)')
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f'$10^{{{int(v)}}}$'))
ax.set_ylim(bottom=0)

ax.text(0.02, 0.96, f'{frac_full:.0%} of positives = 1.0\n(fully retained)',
        transform=ax.transAxes, va='top', ha='left', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', lw=0.8))
ax.text(0.02, 0.04, f'n = {n_domain_pairs:,} domain pairs\n    {n_crosslinks:,} crosslinks',
        transform=ax.transAxes, va='bottom', ha='left', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', lw=0.8))

ax.set_title('Crosslink Retention after Domain Splitting', loc='left')
ax.legend(frameon=False, loc='upper right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig1D_crosslink_retention.pdf', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/fig1D_crosslink_retention.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 3A — ROC of key structural scores + RF models (test set)

ROC curves of the SPOC ESMFOLD, Structural classifier, degree-match baseline and
the structural key scores (ipTM, pDockQ, pDockQ v2, ipSAE d0res), all evaluated
on the same held-out test set.

In [ ]:
# ============================================================
# Figure 3A — ROC curves of key structural scores + RF models (test set)
# ------------------------------------------------------------
# Highlighted: SPOC ESMFOLD / Structural classifier / degree-match
# baseline / ipTM / pDockQ / pDockQ v2 / ipSAE(d0res).
# All computed on the same test set (fair comparison).
# ============================================================
HL = {
    'all_feature_RF':    y_prob_all,                       # RF all-feature
    'structure_only_RF': y_prob_struct,                    # RF structure-only
    'degree_match_RF':   y_prob_degree,                    # degree-match baseline
    'iptm':              df_test['iptm'].values,           # ipTM
    'pdockq':            df_test['pdockq_e'].values,       # pDockQ
    'pdockq2':           df_test['pdockq_e_v2'].values,    # pDockQ v2
    'ipsae_d0res':       df_test['ipsae_d0res_asym'].values,  # ipSAE d0res
}
HL_LABEL = {'all_feature_RF': 'SPOC ESMFOLD', 'structure_only_RF': 'Structural classifier',
            'degree_match_RF': 'degree-match RF', 'iptm': 'ipTM',
            'pdockq': 'pDockQ', 'pdockq2': 'pDockQ v2',
            'ipsae_d0res': 'ipSAE d0res'}
HL_COLORS = {'all_feature_RF': '#1B5E20', 'structure_only_RF': '#0D47A1',
             'degree_match_RF': '#C62828', 'iptm': '#6A1B9A',
             'pdockq': '#00695C', 'pdockq2': '#E65100',
             'ipsae_d0res': '#F57F17'}

plt.figure(figsize=(10, 8), facecolor='white')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.6, label='Random (AUC=0.5)')

print('=== ROC (test set) ===')
for name, score in HL.items():
    mask = np.isfinite(score)
    y, x = y_test[mask], score[mask]
    fpr, tpr, _ = roc_curve(y, x)
    a = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=HL_COLORS[name], linewidth=2.5,
             label=f'{HL_LABEL[name]} (AUC={a:.3f})')
    print(f'  {HL_LABEL[name]:<20s} AUC: {a:.4f}')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC - key structural scores + RF models (test set)', loc='left')
plt.legend(frameon=False, loc='lower right', fontsize=10)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig3A_roc_struct_key_scores.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig3A_roc_struct_key_scores.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 3B — Recall at FDR=5% (1:1 sampling)

For each feature score, the threshold curve of Recall (solid) and FDR (dashed)
averaged over 100 repetitions of 1:1 positive/negative resampling. The FDR=5%
crossing gives the per-score Recall reported in the printed table.

In [ ]:
# ============================================================
# Figure 3B — Recall at FDR=5% (linear interpolation) + threshold curves
# ------------------------------------------------------------
# - 1:1 positive/negative sampling (all positives, negatives downsampled
#   to equal count), repeated N_REP times and averaged.
# - Scores: SPOC ESMFOLD / Structural classifier / pDockQ / pDockQ v2 / ipTM / ipSAE.
# - Solid = Recall, dashed = FDR; FDR=5% reference line shown.
# ============================================================
FDR_TARGET = 0.05
N_REP = 100
rng = np.random.default_rng(42)

SCORE_KEYS_B = list(SCORES.keys())   # all 6 feature scores

def interpolate_first_crossing(grid, rec, fdr, fdr_target=FDR_TARGET):
    """Scan thresholds low->high; at the first crossing of fdr_target,
    linearly interpolate the threshold and recall there. If never crossed,
    fall back to the point closest to the target."""
    t = np.asarray(grid, dtype=float)
    r = np.asarray(rec, dtype=float)
    f = np.asarray(fdr, dtype=float)
    mask = np.isfinite(f) & np.isfinite(r)
    t, r, f = t[mask], r[mask], f[mask]
    if len(f) < 2:
        return np.nan, np.nan
    for i in range(len(f) - 1):
        if (f[i] - fdr_target) * (f[i + 1] - fdr_target) <= 0:
            w = (fdr_target - f[i]) / (f[i + 1] - f[i])
            return float(t[i] + w * (t[i + 1] - t[i])), float(r[i] + w * (r[i + 1] - r[i]))
    j = int(np.argmin(f)) if f[0] > fdr_target else int(np.argmax(f))
    return float(t[j]), float(r[j])

def recall_fdr_curve(score, y, grid):
    """(recall, fdr) on a threshold grid."""
    s = np.asarray(score, dtype=float)
    yy = np.asarray(y, dtype=int)
    mask = np.isfinite(s)
    s, yy = s[mask], yy[mask]
    rec = np.zeros(len(grid))
    fdr = np.full(len(grid), np.nan)
    for i, t in enumerate(grid):
        pred = (s >= t).astype(int)
        tp = int(((pred == 1) & (yy == 1)).sum())
        fp = int(((pred == 1) & (yy == 0)).sum())
        fn = int(((pred == 0) & (yy == 1)).sum())
        rec[i] = tp / (tp + fn) if tp + fn > 0 else 0.0
        fdr[i] = fp / (tp + fp) if tp + fp > 0 else np.nan
    return rec, fdr

pos_idx = np.where(y_test == 1)[0]
neg_idx = np.where(y_test == 0)[0]
n_pos = len(pos_idx)
grid = np.linspace(0, 1, 201)

records = {k: [] for k in SCORE_KEYS_B}
curves = {k: {'rec': [], 'fdr': []} for k in SCORE_KEYS_B}

for rep in range(N_REP):
    neg_sampled = rng.choice(neg_idx, size=n_pos, replace=False)
    idx = np.concatenate([pos_idx, neg_sampled])
    for k in SCORE_KEYS_B:
        r, f = recall_fdr_curve(SCORES[k][idx], y_test[idx], grid)
        curves[k]['rec'].append(r)
        curves[k]['fdr'].append(f)

for k in SCORE_KEYS_B:
    rec_avg = np.mean(curves[k]['rec'], axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        fdr_avg = np.nanmean(np.vstack(curves[k]['fdr']), axis=0)
    thr5, rec5 = interpolate_first_crossing(grid, rec_avg, fdr_avg, FDR_TARGET)
    records[k].append((thr5, FDR_TARGET, rec5))
    curves[k]['rec_avg'] = rec_avg
    curves[k]['fdr_avg'] = fdr_avg

print(f'=== Recall at FDR={FDR_TARGET:.0%} ({N_REP} averaged curves, 1:1 sampling) ===')
print(f'pos n={n_pos}, neg all n={len(neg_idx)}, sampled neg n={n_pos} (1:1)\n')
print(f'{"Feature":<16}{"Threshold":>10}{"FDR":>10}{"Recall":>12}')
for k in SCORE_KEYS_B:
    thr5, fdr5, rec5 = records[k][0]
    print(f'{LABEL_NAME[k]:<16}{thr5:>10.4f}{fdr5:>10.4f}{rec5:>12.4f}')

fig, ax = plt.subplots(figsize=(11, 7), facecolor='white')
for k in SCORE_KEYS_B:
    ax.plot(grid, curves[k]['rec_avg'], color=COLORS[k], ls='-', lw=2.2,
            label=f'{LABEL_NAME[k]} Recall')
    ax.plot(grid, curves[k]['fdr_avg'], color=COLORS[k], ls='--', lw=1.8,
            label=f'{LABEL_NAME[k]} FDR')
ax.axhline(FDR_TARGET, color='black', ls='--', lw=1.5, alpha=0.85,
           label=f'FDR = {FDR_TARGET:.0%}')
ax.set_xlabel('Score Threshold')
ax.set_ylabel('Ratio')
ax.set_title(f'Recall (solid) & FDR (dashed) vs Threshold - 1:1 sampling x {N_REP} reps (mean)', loc='left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=False)
plt.subplots_adjust(right=0.72)
fig.savefig(f'{FIG_DIR}/fig3B_fdr5_recall_threshold.pdf', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/fig3B_fdr5_recall_threshold.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 3C — Recall at FDR=5% across sampling ratios (1:1-1:128)

For each **core score** (SPOC ESMFOLD, Structural classifier, ipSAE, ipTM), the
Recall/FDR vs threshold curves on subsets with positive:negative ratios from
1:1 to 1:128. All scores share the same resampled subsets (fair comparison).

In [ ]:
# ============================================================
# Figure 3C — Recall at FDR=5% across sampling ratios (1:1..1:128)
# ------------------------------------------------------------
# - For each core score (SPOC ESMFOLD / structure RF / ipSAE / ipTM):
#   Recall(solid)/FDR(dashed) vs threshold on subsets with pos:neg = 1:k,
#   k in {1, 4, 16, 64, 128}; FDR=5% recall reported per ratio.
# - Subset construction: n_pos=min(all pos, all neg/k), n_neg=min(all neg, all pos*k).
# - All scores share the SAME sampled subsets (fair comparison).
# - NOTE: also builds `subsets` reused by Fig3D.
# ============================================================
SCORE_KEYS = CORE_SCORE_KEYS          # 4 core scores for Fig3C
FDR_TARGET = 0.05
RATIOS = [1, 4, 16, 64, 128]
N_REP = 100
rng = np.random.default_rng(42)

y_all = df_test['label'].values
pos_idx = np.where(y_all == 1)[0]
neg_idx = np.where(y_all == 0)[0]
n_pos_all = len(pos_idx)
n_neg_all = len(neg_idx)
print(f'Full test: pos {n_pos_all} / neg {n_neg_all} (pos:neg = 1:{n_neg_all/n_pos_all:.2f})')

grid = np.linspace(0, 1, 201)
RATIO_COLORS = {1: '#0D47A1', 4: '#00695C', 16: '#E65100', 64: '#6A1B9A', 128: '#C62828'}

# pre-generate N_REP subset indices per ratio (shared by all scores)
subsets = {}
for k in RATIOS:
    n_pos = int(round(min(n_pos_all, n_neg_all / k)))
    n_neg = int(round(min(n_neg_all, n_pos_all * k)))
    idxs = []
    for _ in range(N_REP):
        pi = pos_idx if n_pos == n_pos_all else rng.choice(pos_idx, size=n_pos, replace=False)
        ni = neg_idx if n_neg == n_neg_all else rng.choice(neg_idx, size=n_neg, replace=False)
        idxs.append(np.concatenate([pi, ni]))
    subsets[k] = (n_pos, n_neg, idxs)

for key in SCORE_KEYS:
    score = SCORES[key]
    print(f'\n======== score [{LABEL_NAME[key]}] ========')
    print(f'{"ratio":>7}{"n_pos":>8}{"n_neg":>8}{"thr@5%":>10}{"recall@5%":>12}')
    curves = {}
    for k in RATIOS:
        n_pos, n_neg, idxs = subsets[k]
        rec_reps, fdr_reps = [], []
        for idx in idxs:
            rec, fdr = recall_fdr_curve(score[idx], y_all[idx], grid)
            rec_reps.append(rec)
            fdr_reps.append(fdr)
        rec_avg = np.mean(rec_reps, axis=0)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            fdr_avg = np.nanmean(np.vstack(fdr_reps), axis=0)
        thr5, rec5 = interpolate_first_crossing(grid, rec_avg, fdr_avg, FDR_TARGET)
        curves[k] = (rec_avg, fdr_avg)
        print(f'1:{k:<6d}{n_pos:>8d}{n_neg:>8d}{thr5:>10.4f}{rec5:>12.4f}')

    fig, ax = plt.subplots(figsize=(11, 7), facecolor='white')
    for k in RATIOS:
        rec_avg, fdr_avg = curves[k]
        ax.plot(grid, rec_avg, color=RATIO_COLORS[k], ls='-', lw=2.2,
                label=f'1:{k} Recall')
        ax.plot(grid, fdr_avg, color=RATIO_COLORS[k], ls='--', lw=1.8,
                label=f'1:{k} FDR')
    ax.axhline(FDR_TARGET, color='black', ls='--', lw=1.5, alpha=0.85,
               label=f'FDR = {FDR_TARGET:.0%}')
    ax.set_xlabel('Score Threshold')
    ax.set_ylabel('Ratio')
    ax.set_title(f'Recall (solid) & FDR (dashed) vs Threshold - {LABEL_NAME[key]} @ 1:1/1:4/1:16/1:64/1:128', loc='left')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=False)
    plt.subplots_adjust(right=0.75)
    fig.savefig(f'{FIG_DIR}/fig3C_fdr5_recall_ratio_{key}.pdf', bbox_inches='tight')
    fig.savefig(f'{FIG_DIR}/fig3C_fdr5_recall_ratio_{key}.png', dpi=300, bbox_inches='tight')
    plt.show()


## Figure 3D — Fixed-threshold recall at 1:128 subset (scatter)

Using the FDR=5% threshold per core score (from the Fig3C averaged curves),
recall is computed on each of the 100 resampled 1:128 subsets. The scatter shows
the per-subset recall distribution with mean +/- std overlaid.

> Run the Figure 3C cell first (this cell reuses its `subsets[128]`).

In [ ]:
# ============================================================
# Figure 3D — Fixed-threshold recall at 1:128 subset (scatter)
# ------------------------------------------------------------
# - Threshold per score = FDR=5% threshold from the averaged curve
#   (same as reported in the Fig3C table).
# - 100 resampled subsets reuse subsets[128] from Fig3C (identical
#   sampling, fair comparison). Recall = TP/(TP+FN) per subset.
# - REQUIRES: run the Fig3C cell first to build `subsets`.
# ============================================================
assert 'subsets' in globals(), 'Please run the Figure 3C cell first (builds subsets)'

RATIO_K = 128
BAR_KEYS = CORE_SCORE_KEYS          # all-feature / structure / ipSAE / ipTM
FDR_TARGET = 0.05

n_pos_128, n_neg_128, idxs_128 = subsets[RATIO_K]
print(f'1:{RATIO_K} subset: n_pos={n_pos_128}, n_neg={n_neg_128} ({len(idxs_128)} samples)')

# 1) fixed threshold per score = averaged-curve FDR=5% crossing
fixed_thr = {}
for k in BAR_KEYS:
    rec_reps, fdr_reps = [], []
    for idx in idxs_128:
        r, f = recall_fdr_curve(SCORES[k][idx], y_all[idx], grid)
        rec_reps.append(r)
        fdr_reps.append(f)
    rec_avg = np.mean(rec_reps, axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        fdr_avg = np.nanmean(np.vstack(fdr_reps), axis=0)
    thr, _ = interpolate_first_crossing(grid, rec_avg, fdr_avg, FDR_TARGET)
    fixed_thr[k] = thr
    print(f'  fixed threshold {LABEL_NAME[k]:<18s} = {thr:.4f}')

# 2) recall per resampled subset at the fixed threshold
def recall_at_threshold(score, y, thr):
    s = np.asarray(score, dtype=float)
    yy = np.asarray(y, dtype=int)
    mask = np.isfinite(s)
    s, yy = s[mask], yy[mask]
    pred = (s >= thr).astype(int)
    tp = int(((pred == 1) & (yy == 1)).sum())
    fp = int(((pred == 1) & (yy == 0)).sum())
    fn = int(((pred == 0) & (yy == 1)).sum())
    rec = tp / (tp + fn) if tp + fn > 0 else 0.0
    fdr = fp / (tp + fp) if tp + fp > 0 else np.nan
    return rec, fdr

rec_fixed = {k: [] for k in BAR_KEYS}
fdr_fixed = {k: [] for k in BAR_KEYS}
for k in BAR_KEYS:
    for idx in idxs_128:
        rec, fdr = recall_at_threshold(SCORES[k][idx], y_all[idx], fixed_thr[k])
        rec_fixed[k].append(rec)
        fdr_fixed[k].append(fdr)
    rec_fixed[k] = np.asarray(rec_fixed[k])
    fdr_fixed[k] = np.asarray(fdr_fixed[k])
    print(f'  {LABEL_NAME[k]:<18s}: recall = {rec_fixed[k].mean():.4f} +/- {rec_fixed[k].std():.4f} '
          f'(median {np.median(rec_fixed[k]):.4f}); actual FDR = {np.nanmean(fdr_fixed[k]):.4f}')

bar_names = [LABEL_NAME[k] for k in BAR_KEYS]
x_positions = np.arange(len(bar_names))

fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

y_max = 0.0
for j, k in enumerate(BAR_KEYS):
    recs = rec_fixed[k]
    y_max = max(y_max, recs.max())
    jitter = rng.uniform(-0.18, 0.18, size=len(recs))
    ax.scatter(x_positions[j] + jitter, recs, s=20, alpha=0.6,
               color=COLORS[k], edgecolor='none', zorder=2)
    ax.errorbar(x_positions[j], recs.mean(), yerr=recs.std(),
                fmt='o', color='black', markersize=6, capsize=4,
                elinewidth=1.5, markeredgewidth=1.0, zorder=3)
ax.set_xticks(x_positions)
ax.set_xticklabels(bar_names)
ax.set_ylabel('Recall (fixed FDR=5% threshold)')
ax.set_ylim(0, max(0.05, 1.15 * y_max))
ax.set_title(f'Recall @ fixed FDR={FDR_TARGET:.0%} threshold - 1:{RATIO_K} subset x {len(idxs_128)} samples', loc='left')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig3D_recall_fixedthr_scatter_1to{RATIO_K}.pdf', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/fig3D_recall_fixedthr_scatter_1to{RATIO_K}.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 4A / 4B — Gini importance

Mean decrease in impurity (Gini importance) of every feature.
- **4A**: SPOC ESMFOLD
- **4B**: Structural classifier

In [ ]:
# ============================================================
# Figure 4A / 4B — Gini importance (mean decrease in impurity)
# ------------------------------------------------------------
# 4A: SPOC ESMFOLD, 4B: Structural classifier.
# Gini importance from rf_permutation_importance_*.csv (gini_importance col).
# ============================================================
MODEL_CSVS = [
    ('SPOC ESMFOLD',            f'{DATA}/rf_permutation_importance_allfeat.csv',    '#1B5E20'),
    ('Structural classifier',   f'{DATA}/rf_permutation_importance_structfeat.csv', '#0D47A1'),
]

for tag, csv_path, color in MODEL_CSVS:
    tag_label = tag
    df = pd.read_csv(csv_path).sort_values('gini_importance', ascending=False).reset_index(drop=True)
    feats = [f.replace('_', ' ') for f in df['feature'].tolist()]
    x = np.arange(len(feats))
    fig_w = max(10, 0.35 * len(feats))

    fig, ax = plt.subplots(figsize=(fig_w, 6), facecolor='white')
    ax.bar(x, df['gini_importance'], color=color, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(feats, rotation=90, fontsize=8)
    ax.set_ylabel('Gini importance')
    ax.set_title(f'Gini importance - {tag_label}', loc='left')
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/fig4_{tag}_gini.pdf', bbox_inches='tight')
    fig.savefig(f'{FIG_DIR}/fig4_{tag}_gini.png', dpi=300, bbox_inches='tight')
    plt.show()

print(f'Done: saved Fig4A/4B to {FIG_DIR}')


## Supplementary Figure — AUPR (Precision-Recall) on test set

Precision-Recall curves with AUPR of the three models (SPOC ESMFOLD / Structural classifier
/ degree-match baseline). The dashed line is the positive ratio (expected AUPR of a
random classifier).

In [ ]:
# ============================================================
# Supplementary Figure — AUPR (Precision-Recall) on test set
# ------------------------------------------------------------
# all-feature / structure-only / degree-match baseline.
# Baseline = positive ratio (expected AUPR of a random classifier).
# ============================================================
def get_pr(y_true, scores):
    mask = np.isfinite(scores)
    p, r, _ = precision_recall_curve(y_true[mask], scores[mask])
    ap = average_precision_score(y_true[mask], scores[mask])
    return p, r, ap

prs = {
    'SPOC ESMFOLD':    get_pr(y_test, y_prob_all),
    'Structural classifier': get_pr(y_test, y_prob_struct),
    'degree-match':   get_pr(y_test, y_prob_degree),
}
colors = {'SPOC ESMFOLD': '#1B5E20', 'Structural classifier': '#0D47A1', 'degree-match': '#C62828'}

pos_ratio = y_test.mean()
print(f'Positive ratio (baseline AUPR): {pos_ratio:.4f}')
for name, pr in prs.items():
    print(f'{name:15s} AUPR: {pr[2]:.4f}')

plt.figure(figsize=(10, 8), facecolor='white')
plt.axhline(pos_ratio, color='gray', linestyle='--', linewidth=1.5, alpha=0.7,
            label=f'Baseline (pos ratio={pos_ratio:.3f})')
for name, pr in prs.items():
    plt.plot(pr[1], pr[0], color=colors[name], linewidth=2.5,
             label=f'{name} (AUPR={pr[2]:.3f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall (AUPR) on Test Set - 3 models', loc='left')
plt.legend(frameon=False, loc='upper right')
plt.xlim([0, 1.02])
plt.ylim([0, 1.02])
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/supp_aupr_test_3models.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/supp_aupr_test_3models.png', dpi=300, bbox_inches='tight')
plt.show()


## Supplementary Figure — RF hyperparameter sweep (3-fold CV AUPR)

3-fold CV AUPR heatmaps over `n_estimators` x `max_depth` (one panel per
`min_samples_split`) for the two classifiers:
- **SPOC ESMFOLD** (all features)
- **Structural classifier** (structure-only)

The underlying scan is performed by `scripts/param_sweep_aupr_heatmap.py`;
this cell reads the pre-computed results from `data/analysis/` and redraws the
heatmaps.

In [ ]:
# ============================================================
# Supplementary Figure - RF hyperparameter sweep (3-fold CV AUPR)
# ------------------------------------------------------------
# Grid search over n_estimators x max_depth x min_samples_split for
# the two classifiers (SPOC ESMFOLD & Structural classifier).
# Data: pre-computed results in data/analysis/param_sweep_aupr_results_{all,struct}.csv
# (generated by scripts/param_sweep_aupr_heatmap.py; one row per parameter
#  combination with the 3-fold CV AUPR of each fold).
# ============================================================
SWEEP = [
    ('SPOC ESMFOLD',            f'{DATA}/param_sweep_aupr_results_all.csv'),
    ('Structural classifier',   f'{DATA}/param_sweep_aupr_results_struct.csv'),
]
N_EST  = [200, 400, 600, 1000]
DEPTH  = [8, 12, 16, 20, 24]
SPLIT  = [2, 5, 10, 20]

for name, csv_path in SWEEP:
    res = pd.read_csv(csv_path)
    vmin, vmax = float(res['mean'].min()), float(res['mean'].max())

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor='white')
    for ax, ms in zip(axes.ravel(), SPLIT):
        sub = res[res['min_samples_split'] == ms]
        mat = sub.pivot(index='max_depth', columns='n_estimators',
                        values='mean').reindex(index=DEPTH, columns=N_EST).astype(float)
        std = sub.pivot(index='max_depth', columns='n_estimators',
                        values='std').reindex(index=DEPTH, columns=N_EST).astype(float)

        im = ax.imshow(mat.values, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(N_EST)))
        ax.set_xticklabels([str(n) for n in N_EST])
        ax.set_yticks(range(len(DEPTH)))
        ax.set_yticklabels([str(d) for d in DEPTH])
        ax.set_xlabel('n_estimators')
        ax.set_ylabel('max_depth')
        ax.set_title(f'min_samples_split = {ms}', loc='left', fontsize=11)

        for i in range(len(DEPTH)):
            for j in range(len(N_EST)):
                c = 'white' if mat.values[i, j] > (vmin + vmax) / 2 else 'black'
                ax.text(j, i, f"{mat.values[i, j]:.3f}
±{std.values[i, j]:.3f}",
                        ha='center', va='center', fontsize=8, color=c)

    fig.colorbar(im, ax=axes, shrink=0.8, label='mean AUPR')
    fig.suptitle(f'3-fold CV AUPR hyperparameter sweep - {name}', fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    tag = name.replace(' ', '_')
    fig.savefig(f'{FIG_DIR}/supp_param_sweep_aupr_{tag}.pdf', bbox_inches='tight')
    fig.savefig(f'{FIG_DIR}/supp_param_sweep_aupr_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show()

print(f'Done: supplementary hyperparameter-sweep heatmaps saved to {FIG_DIR}')
